In [14]:
import os
print(os.getcwd())
print(os.listdir())
from sklearn.cluster import DBSCAN
from thefuzz import fuzz
import pandas as pd
import numpy as np

/users/ctlin/fooddesertproject/urbanFarmData/deduplicated_farms
['cleaned_dallas_geojson.csv', 'deduplicating.ipynb', 'usda_csa.csv', 'usda_ofm.csv', '.ipynb_checkpoints', 'converting_to_csv.ipynb', 'deduplicated_dallas_farms.csv', 'deduplicated_travis_farms.csv', 'deduplicated_tarrant_farms.csv', 'deduplicated_harris_farms.csv']


In [11]:
county = "bexar"
fips ="029"
city = "san_antonio"

# load all the different csv files 

# google api data 
google = pd.read_csv(f"../google_maps_api/{county}_county_farms.csv")
print(f"Google: {len(google)} rows, columns: {list(google.columns)}")
google = google.rename(columns={
    "Name" : "name",
    "Address" : "address",
    "Latitude": "lat",
    "Longitude": "lon"
})

# local harvest
local_harvest = pd.read_csv(f"../local_harvest_geocoded/{city}_geocoded.csv")
print(f"Local Harvest: {len(local_harvest)} rows, columns: {list(local_harvest.columns)}")
local_harvest = local_harvest.rename(columns={
    "farm_name": "name",
    "latitude": "lat",
    "longitude": "lon",
    "street_address": "address"
})

# open street maps- will need to filter by county later by name 
osm = pd.read_csv("../osm_data/urban_farms_with_addresses.csv")
print(f"OSM (full): {len(osm)} rows")
osm = osm.rename(columns={
    "latitude": "lat",
    "longitude": "lon"
})

# usda data - will need to sort by county later, by FIPS number 
csa = pd.read_csv("usda_csa.csv")
print(f"csa (full): {len(csa)} rows")
csa.rename(columns={
    "latitude": "lat",
    "longitude": "lon"
})

ofm = pd.read_csv("usda_ofm.csv")
print(f"ofm (full): {len(ofm)} rows")
ofm.rename(columns={
    "latitude": "lat",
    "longitude": "lon"
})

# texas local food data - need to sort by county name 
tx = pd.read_csv("../tx_local_food_direct.csv")
print(f"tx (full): {len(tx)} rows")
tx.rename(columns={
    "farm_name": "name",
    "latitude": "lat",
    "longitude": "lon",
    "street_address":"address"
})
    
# farmers market local data -need to sort by county name
market = pd.read_csv("../farmers_market_local_list.csv")
print(f"market (full): {len(market)} rows")
market.rename(columns={
    "farm_name":"name",
    "latitude": "lat",
    "longitude": "lon",
    "street_address":"address"
})
# original dallas data 
# dallas = pd.read_csv("cleaned_dallas_geojson.csv")
# dallas.rename(columns={
#     "latitude": "lat",
#     "longitude": "lon"
# })
# print(f"dallas : {len(dallas)} rows")

Google: 146 rows, columns: ['Name', 'Address', 'Latitude', 'Longitude', 'Website', 'Google Categories', 'Place ID', 'Status']
Local Harvest: 48 rows, columns: ['farm_name', 'profile_url', 'latitude', 'longitude', 'street_address']
OSM (full): 47 rows
csa (full): 14 rows
ofm (full): 14 rows
tx (full): 27 rows
market (full): 11 rows


,name,address,city,county,zip_code,lat,lon,website,date_collected,geometry,...,LSADC,FUNCSTAT,COUNTYCC,AREALAND,AREAWATER,OBJECTID,CENTLAT,CENTLON,INTPTLAT,INTPTLON
0,Dragon City Farms,2435 Walnut Ridge Street,"Dallas, TX",Dallas,75229,32.894300,-96.885600,https://dragoncityfarms.com,6/29/26,POINT (-96.8856 32.8943),...,6,A,H1,2261553244,91821724,1563,32.766485,-96.777765,32.766987,-96.778424
1,Gardeners in Community Development,901 Greenbriar Ln,"Richardson, TX",Dallas/Collin,75080,32.955400,-96.745600,https://www.facebook.com/gardendallas/,6/29/26,POINT (-96.7456 32.9554),...,6,A,H1,2261553244,91821724,1563,32.766485,-96.777765,32.766987,-96.778424
2,GROW North Texas,1451 John West Rd,"Dallas, TX",Dallas,75228,32.812300,-96.657800,http://www.grownorthtexas.org,6/29/26,POINT (-96.6578 32.8123),...,6,A,H1,2261553244,91821724,1563,32.766485,-96.777765,32.766987,-96.778424
3,Restorative Farms,4500 Todd Street,"Dallas, TX",Dallas,75210,32.768100,-96.732500,https://www.restorativefarms.org,6/29/26,POINT (-96.7325 32.7681),...,6,A,H1,2261553244,91821724,1563,32.766485,-96.777765,32.766987,-96.778424
4,Sustainable Stems,7017 Pickrell Dr,"Dallas, TX",Dallas,75227,32.790031,-96.691947,https://sustainablestemsdfw.com,6/29/26,POINT (-96.69194687 32.79003147),...,6,A,H1,2261553244,91821724,1563,32.766485,-96.777765,32.766987,-96.778424
5,Opal's Farm,2500 Lasalle St,"Fort Worth, TX",Tarrant,76111,32.756594,-97.311558,https://www.facebook.com/opalsfarm/,6/29/26,POINT (-97.31155814 32.75659382),...,6,A,H1,2241376178,99392439,2656,32.771835,-97.291132,32.772119,-97.291224
6,Joe's Microgreens,7206 Shelton Rd,"Austin, TX",Travis,78725,30.261774,-97.666163,https://joesmicrogreens.com/,6/29/26,POINT (-97.66616341 30.26177419),...,6,A,H1,2575299585,79506870,2420,30.334360,-97.781817,30.239526,-97.691053
7,Urban Roots,7651 Delwau Ln,"Austin, TX",Travis,78725,30.263129,-97.664206,https://urbanrootsatx.org/,6/29/26,POINT (-97.66420595 30.26312865),...,6,A,H1,2575299585,79506870,2420,30.334360,-97.781817,30.239526,-97.691053
8,Tecolote Farm,"16301 Decker Lake Rd,","Manor, TX",Travis,78653,30.257773,-97.563959,http://www.tecolotefarm.net,6/29/26,POINT (-97.56395878 30.25777345),...,6,A,H1,2575299585,79506870,2420,30.334360,-97.781817,30.239526,-97.691053
9,Ivy Leaf Farms,4506 Fuqua St,"Houston, TX",Harris,77048,29.620000,-95.358200,https://www.ivyleaffarms.com/,6/29/26,POINT (-95.3582 29.62),...,6,A,H1,4421991136,180517819,2814,29.857333,-95.392002,29.857273,-95.393037


In [3]:
# filtering the csv which contain all counties 
county_osm = osm[osm["county"] == county.capitalize()]
print(f"OSM filtered to '{county.capitalize()}': {len(county_osm)} rows")

county_csa = csa[csa["county"].astype(str) == fips]
print(f"csa filtered to '{county.capitalize()}': {len(county_csa)} rows")

county_ofm = ofm[ofm["county"].astype(str) == fips]
print(f"ofm filtered to '{county.capitalize()}': {len(county_ofm)} rows")

county_tx = tx[tx["county"] == county.capitalize()]
print(f"tx filtered to '{county.capitalize()}': {len(county_tx)} rows")

county_market = market[market["county"] == county.capitalize()]
print(f"market filtered to '{county.capitalize()}': {len(county_market)} rows")

OSM filtered to 'Bexar': 5 rows
csa filtered to 'Bexar': 0 rows
ofm filtered to 'Bexar': 0 rows
tx filtered to 'Bexar': 4 rows
market filtered to 'Bexar': 0 rows


In [12]:
# --- Combine all sources ---
combined = pd.concat(
    [google, local_harvest, county_osm, county_csa, county_ofm, county_tx, county_market],
    ignore_index=True
)
print(f"\n Combined (before dedup): {len(combined)} rows")
print(f"   Columns: {list(combined.columns)}")


 Combined (before dedup): 203 rows
   Columns: ['name', 'address', 'lat', 'lon', 'Website', 'Google Categories', 'Place ID', 'Status', 'profile_url', 'osm_id', 'fclass', 'county', 'COUNTYFP', 'house_number', 'street', 'city', 'zipcode', 'full_address', 'Unnamed: 0', 'latitude', 'longitude', 'oid', 'geoid', 'state', 'farm_id', 'farm_name', 'street_address', 'zip_code', 'latitude ', 'date_collected', 'website', 'geometry', 'index_right', 'MTFCC', 'OID', 'GEOID', 'STATE', 'COUNTY', 'COUNTYNS', 'BASENAME', 'NAME', 'LSADC', 'FUNCSTAT', 'COUNTYCC', 'AREALAND', 'AREAWATER', 'OBJECTID', 'CENTLAT', 'CENTLON', 'INTPTLAT', 'INTPTLON']


In [60]:
# print(combined.columns[combined.columns.duplicated()].tolist())
# cols = list(combined.columns)
# seen = {}
# new_cols = []
# for c in cols:
#     if c in seen:
#         seen[c] += 1
#         new_cols.append(f"{c}_{seen[c]}")
#     else:
#         seen[c] = 0
#         new_cols.append(c)
# combined.columns = new_cols

# print(combined.columns[combined.columns.duplicated()].tolist())  # should be empty now
# print([c for c in combined.columns if "lat" in c.lower() or "lon" in c.lower()])

# combined["lat"] = (
#     combined["lat"]
#     .fillna(combined["latitude"])
#     .fillna(combined["latitude_1"])
#     .fillna(combined["CENTLAT"])
#     .fillna(combined["INTPTLAT"])
# )

# combined["lon"] = (
#     combined["lon"]
#     .fillna(combined["longitude"])
#     .fillna(combined["CENTLON"])
#     .fillna(combined["INTPTLON"])
# )

# combined["name"] = combined["name"].fillna(combined.get("NAME"))

# print(f"Missing lat/lon after consolidation: {combined['lat'].isna().sum()}")

# missing = combined[combined["lat"].isna()]
# print(missing[["name"]].to_string())

[]
[]
['lat', 'lon', 'latitude', 'longitude', 'latitude ', 'CENTLAT', 'CENTLON', 'INTPTLAT', 'INTPTLON']


KeyError: 'latitude_1'

In [15]:
# Drop rows missing lat/lon (can't cluster them)
combined = combined.dropna(subset=["lat", "lon"]).copy()
print(f"Rows after dropping missing coords: {len(combined)}")

# Now redo the meters conversion and DBSCAN
lat_rad = np.radians(combined["lat"].mean())
coords_meters = combined[["lat", "lon"]].copy()
coords_meters["lat_m"] = combined["lat"] * 111320
coords_meters["lon_m"] = combined["lon"] * (111320 * np.cos(lat_rad))

clustering = DBSCAN(eps=500, min_samples=1).fit(coords_meters[["lat_m", "lon_m"]])
combined["cluster_id"] = clustering.labels_
print(f"🗺️  DBSCAN found {combined['cluster_id'].nunique()} geographic clusters")

Rows after dropping missing coords: 186
🗺️  DBSCAN found 173 geographic clusters


In [16]:
import numpy as np

# Simple conversion: 1 degree lat ≈ 111,320m, 1 degree lon ≈ 111,320 * cos(lat)
lat_rad = np.radians(combined["lat"].mean())
coords_meters = combined[["lat", "lon"]].copy()
coords_meters["lat_m"] = combined["lat"] * 111320
coords_meters["lon_m"] = combined["lon"] * (111320 * np.cos(lat_rad))

# --- 2. DBSCAN to group nearby farms ---
clustering = DBSCAN(eps=500, min_samples=1).fit(coords_meters[["lat_m", "lon_m"]])
combined["cluster_id"] = clustering.labels_
print(f"🗺️  DBSCAN found {combined['cluster_id'].nunique()} geographic clusters")

# --- 3. Clean names for fuzzy matching ---
combined["clean_name"] = (
    combined["name"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.strip()
)

# --- 4. Fuzzy match within each cluster ---
indices_to_drop = set()
fuzzy_threshold = 85  # ← lower = more aggressive, raise if removing too much

for cluster_id, group in combined.groupby("cluster_id"):
    if len(group) < 2:
        continue

    records = list(group["clean_name"].items())
    print(f"\n🔍 Cluster {cluster_id} ({len(records)} farms): {[n for _, n in records]}")

    for i in range(len(records)):
        idx1, name1 = records[i]
        if idx1 in indices_to_drop:
            continue
        for j in range(i + 1, len(records)):
            idx2, name2 = records[j]
            if idx2 in indices_to_drop:
                continue
            score = fuzz.token_set_ratio(str(name1), str(name2))
            if score >= fuzzy_threshold:
                print(f"   ✂️  DROP '{name2}' (matches '{name1}', score={score})")
                indices_to_drop.add(idx2)

# --- 5. Drop duplicates and clean up ---
deduped = combined.drop(index=list(indices_to_drop)).drop(columns=["cluster_id", "clean_name"])
print(f"\n✅ After fuzzy dedup: {len(deduped)} rows  ({len(combined) - len(deduped)} removed)")

deduped.to_csv(f"deduplicated_{county}_farms.csv", index=False)
print(f"💾 Saved to deduplicated_{county}_farms.csv")


🗺️  DBSCAN found 173 geographic clusters

🔍 Cluster 3 (2 farms): ['pearl farmers market', 'shudde ranch beef  parker creek ranch']

🔍 Cluster 7 (2 farms): ['make ready market', 'elsewhere garden bar  kitchen']

🔍 Cluster 8 (2 farms): ['chicho boys fruit market', 'alamo farms']

🔍 Cluster 12 (2 farms): ['market plaza center', 'san antonio farmers market']

🔍 Cluster 23 (2 farms): ['rainbow gardens', 'ryans happy hens']

🔍 Cluster 36 (2 farms): ['the greenhouse at vintage heart farm', 'vintage heart farm']
   ✂️  DROP 'vintage heart farm' (matches 'the greenhouse at vintage heart farm', score=100)

🔍 Cluster 58 (2 farms): ['bonton farms', 'the farmers market at bonton farms']
   ✂️  DROP 'the farmers market at bonton farms' (matches 'bonton farms', score=100)

🔍 Cluster 70 (3 farms): ['good ground flower farm', 'spice farm', 'good ground farm']
   ✂️  DROP 'good ground farm' (matches 'good ground flower farm', score=100)

🔍 Cluster 90 (2 farms): ['jefferson community garden', 'jefferson 